# Chess Transformer Training Notebook

This notebook downloads a Lichess PGN dataset and trains the chess transformer model.


## 1. Setup and Imports


In [ ]:
import sys
import os
from pathlib import Path

# Add project root to path (one level up from notebooks/)
PROJECT_ROOT = str(Path(__file__).resolve().parent.parent) if '__file__' in dir() else str(Path.cwd().parent)
sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

import torch
from src.download import download_file, construct_lichess_url
from src.data import pgn_to_samples, ChessDataset, get_optimal_workers
from src.model import create_model
from src.train import train, train_epoch, validate, _unwrap_model, DEVICE
from torch.utils.data import DataLoader, random_split
import torch.nn as nn
import torch.optim as optim


## 2. Download Lichess PGN Dataset


In [ ]:
# Configuration
DATE = "2025-10"  # Change this to the date you want (YYYY-MM format)
# Or use a full URL:
# URL = "https://database.lichess.org/standard/lichess_db_standard_rated_2025-10.pgn.zst"

# Construct URL from date
url = construct_lichess_url(DATE)
output_dir = "data"
os.makedirs(output_dir, exist_ok=True)

# Extract filename from URL
filename = f"lichess_db_standard_rated_{DATE}.pgn.zst"
output_path = os.path.join(output_dir, filename)

print(f"Downloading: {url}")
print(f"Output: {output_path}")

# Download file
if os.path.exists(output_path):
    print(f"\nFile already exists: {output_path}")
    print("Skipping download. Delete file to re-download.")
else:
    success = download_file(url, output_path, resume=True)
    if not success:
        print("Download failed or interrupted. You can resume by running this cell again.")
    else:
        print(f"\n✅ Download complete: {output_path}")


## 3. Training Configuration


In [ ]:
# Training hyperparameters
config = {
    'pgn_file': output_path,  # Use downloaded file
    'max_games': 10000,  # Limit games for faster training (None for all)
    'min_rating': None,  # Minimum player rating (None for all)
    'batch_size': 32,
    'epochs': 5,
    'lr': 1e-4,
    'hidden_dim': 256,
    'n_layers': 6,
    'n_heads': 8,
    'val_split': 0.1,  # 10% validation
    'parse_workers': 4,  # Parallel parsing workers (set to 1 for sequential)
    'num_workers': None,  # Auto-detect DataLoader workers
    'use_cache': True,
    'cache_dir': 'cache',
    'save_dir': 'models',
    'save_prefix': 'minichess_transformer'
}

print("Training Configuration:")
for key, value in config.items():
    print(f"  {key}: {value}")


## 4. Load and Prepare Data


In [ ]:
print(f"Using device: {DEVICE}")
print("\nLoading data...")

# Auto-detect num_workers
if config['num_workers'] is None:
    config['num_workers'] = get_optimal_workers()
print(f"DataLoader workers: {config['num_workers']}")

# Parse PGN file into contiguous tensors
tensor_dict, move_encoder = pgn_to_samples(
    config['pgn_file'],
    max_games=config['max_games'],
    min_rating=config['min_rating'],
    use_cache=config['use_cache'],
    cache_dir=config['cache_dir'],
    num_parse_workers=config['parse_workers']
)

n_samples = tensor_dict['piece_ids'].shape[0]
print(f"\nLoaded {n_samples:,} training samples")
print(f"Move vocabulary size: {move_encoder.get_vocab_size()}")


In [ ]:
# Create dataset and split
full_dataset = ChessDataset(tensor_dict)
total_size = len(full_dataset)
val_size = int(total_size * config['val_split'])
train_size = total_size - val_size

train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

# Create data loaders (optimised for CUDA multi-GPU setups)
pin_memory = (DEVICE == "cuda")
persistent_workers = (config['num_workers'] > 0)

loader_kwargs = {
    'num_workers': config['num_workers'],
    'pin_memory': pin_memory,
    'persistent_workers': persistent_workers,
}
if config['num_workers'] > 0:
    loader_kwargs['prefetch_factor'] = 4

train_loader = DataLoader(
    train_dataset,
    batch_size=config['batch_size'],
    shuffle=True,
    drop_last=True,
    **loader_kwargs,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=config['batch_size'],
    shuffle=False,
    **loader_kwargs,
)

print(f"Train samples: {train_size:,}, Val samples: {val_size:,}")


## 5. Create Model


In [ ]:
# Get move vocabulary size
move_vocab_size = move_encoder.get_vocab_size()

# Create model
model = create_model(
    vocab_size=14,  # Fixed: 12 pieces + empty + padding
    hidden_dim=config['hidden_dim'],
    n_layers=config['n_layers'],
    n_heads=config['n_heads'],
    move_vocab=move_vocab_size
).to(DEVICE)

num_params = sum(p.numel() for p in model.parameters())
print(f"Model created with {num_params:,} parameters")
print(f"Move vocabulary size: {move_vocab_size}")

# Wrap in DataParallel if multiple GPUs are available
n_gpus = torch.cuda.device_count()
if DEVICE == "cuda" and n_gpus > 1:
    print(f"Wrapping model in DataParallel across {n_gpus} GPUs: "
          + ", ".join(torch.cuda.get_device_name(i) for i in range(n_gpus)))
    model = nn.DataParallel(model)


## 6. Train Model


In [ ]:
# Create optimizer
optimizer = optim.AdamW(model.parameters(), lr=config['lr'], weight_decay=1e-5)

# Prepare move encoder state for saving
move_encoder_state = {
    'move_to_id': move_encoder.move_to_id,
    'id_to_move': move_encoder.id_to_move,
    'next_move_id': move_encoder.next_move_id
}

# Train
print("Starting training...")
train(
    model, train_loader, val_loader, optimizer, config['epochs'], DEVICE,
    save_dir=config['save_dir'],
    save_prefix=config['save_prefix'],
    move_encoder_state=move_encoder_state
)

print("\n✅ Training complete!")


## 7. Quick Test (Optional)


In [ ]:
# Test inference on a simple position
from src.inference import predict_move_from_legal, evaluate_position, load_model
import chess

# Load best model
checkpoint_path = os.path.join(config['save_dir'], f"{config['save_prefix']}_best.pt")
if os.path.exists(checkpoint_path):
    test_model, info = load_model(checkpoint_path)
    board = chess.Board()
    
    print("Testing on starting position:")
    print(board)
    
    # Evaluate position
    value = evaluate_position(test_model, board)
    print(f"\nPosition evaluation: {value:.4f} (from White's perspective)")
    
    # Predict moves
    move_encoder = info.get('move_encoder')
    if move_encoder:
        moves = predict_move_from_legal(test_model, board, move_encoder=move_encoder, top_k=5)
        print("\nTop 5 predicted moves:")
        for i, (move, prob) in enumerate(moves, 1):
            print(f"  {i}. {move.uci()}: {prob:.4f}")
    else:
        print("\nWarning: Move encoder not found in checkpoint")
else:
    print(f"Best model not found at {checkpoint_path}")
